# Pokemon Battle Prediction - FDS 2026

This notebook clones our GitHub repository and uses the `src` package to
reproduce the three selected submissions. All the code lives in the repository;
here we only call it.

Repository: https://github.com/Hiperversum/Pokemon-Challange-ML

The hyperparameters of the tuned models and the members of the ensembles are read
from `data/models/tuned_params.json` and `data/models/ensembles.json` in the repository.

## 1. Clone the repository

In [ ]:
!git clone https://github.com/Hiperversum/Pokemon-Challange-ML.git

## 2. Install the required libraries

In [ ]:
!cd Pokemon-Challange-ML && pip install -q -r requirements.txt

## 3. Update the system path

In [ ]:
import sys

repo_path = "/kaggle/working/Pokemon-Challange-ML"
if repo_path not in sys.path:
    sys.path.append(repo_path)

## 4. Import our modules and build the features

In [ ]:
from src.utility.config.config_paths import get_data_dir
from src.evaluation.validation import compare_models, cross_validate
from src.models.core.model_builder import build_ensemble, build_model, build_tuned_model
from src.features.load_dataset import load_dataset
from src.models.submission import fit_and_predict, make_submission

print("Data folder:", get_data_dir())
X_train, y_train, X_test = load_dataset()
print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Validation

All decisions are taken with stratified 5-fold cross-validation on the training
set. The table below compares the base models on the full feature set.

In [ ]:
base_models = {name: build_model(name) for name in ["logreg", "knn", "tree", "random_forest", "xgboost"]}
comparison = compare_models(base_models, X_train.values, y_train.values)
comparison

## Submission 1 - Soft voting of four models

Average of the predicted probabilities of four different kinds of model: L1 logistic
regression (linear), k-nearest neighbors (based on distances), random forest (bagging
of trees) and XGBoost (boosting of trees). We chose the members by computing the
cross-validation accuracy of every combination of our models: kNN is the weakest model
alone, but it is in all the best combinations, because its errors are different from
the errors of the other models. Cross-validation accuracy 0.684, the best of all the candidates.

In [ ]:
model = build_ensemble("voting")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_1_voting.csv"))

## Submission 2 - L1 logistic regression

Logistic regression with an L1 penalty (C = 0.03, found by Optuna) on
standardized features. The penalty sets to zero the weights of the features that
do not help, so the model keeps about 250 of the 583 features. It is simple, very
fast and as accurate as the ensembles (cross-validation accuracy 0.682), and it is
very different from the tree models of the other submissions.

In [ ]:
model = build_tuned_model("logreg")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_2_logreg_l1.csv"))

## Submission 3 - Stacking of the same four models

The same four models (logistic regression, kNN, random forest and XGBoost). Instead of
averaging their probabilities, a logistic regression is trained on their out-of-fold
predictions and learns how much each model counts. Cross-validation accuracy 0.6825.

In [ ]:
model = build_ensemble("stacking")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_3_stacking.csv"))